In [0]:
%sql
create schema if not exists mobility.bronze;
create schema if not exists mobility.silver;
create schema if not exists mobility.gold;


In [0]:
spark.table("mobility.bronze.yellow_trips_raw").printSchema()
spark.table("mobility.bronze.zone_lookup_raw").printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- _rescued_data: string (nullable = true)

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable =

In [0]:
bronze = spark.table("mobility.bronze.yellow_trips_raw").count()
clean = spark.table("mobility.silver.trips_clean").count()
bad = spark.table("mobility.silver.trips_quarantine").count()
print(bronze, clean, bad, clean + bad == bronze)

2964624 2926310 38314 True


In [0]:
spark.table("mobility.silver.trips_quarantine").groupBy("reject_reason").count().show()

+--------------------+-----+
|       reject_reason|count|
+--------------------+-----+
|DROPOFF_NOT_AFTER...|  870|
|       NEGATIVE_FARE|37444|
+--------------------+-----+



In [0]:
spark.table("mobility.silver.trips_quarantine") \
    .filter("reject_reason = 'NEGATIVE_FARE'") \
    .groupBy("payment_type").count().show()
    

+------------+-----+
|payment_type|count|
+------------+-----+
|           4|21406|
|           2| 8208|
|           3| 5736|
|           1|   28|
|           0| 2066|
+------------+-----+



In [0]:
spark.table("mobility.gold.trips_by_zone").orderBy("total_trips", ascending=False).show(10)
spark.table("mobility.gold.trips_by_hour").orderBy("pickup_hour").show(24)
spark.table("mobility.gold.monthly_summary").show()


+--------------------+---------+-----------+--------+
|                Zone|  Borough|total_trips|avg_fare|
+--------------------+---------+-----------+--------+
|      Midtown Center|Manhattan|     141725|   15.58|
|Upper East Side S...|Manhattan|     141244|   12.42|
|         JFK Airport|   Queens|     141142|   62.47|
|Upper East Side N...|Manhattan|     135307|   12.92|
|        Midtown East|Manhattan|     105445|   15.14|
|Times Sq/Theatre ...|Manhattan|     104594|    18.1|
|Penn Station/Madi...|Manhattan|     103256|   16.17|
| Lincoln Square East|Manhattan|     102960|   13.72|
|   LaGuardia Airport|   Queens|      88568|   42.29|
|Upper West Side S...|Manhattan|      87633|    13.7|
+--------------------+---------+-----------+--------+
only showing top 10 rows
+-----------+-----------+
|pickup_hour|total_trips|
+-----------+-----------+
|          0|      77655|
|          1|      52668|
|          2|      36767|
|          3|      24234|
|          4|      16292|
|          

In [0]:
%sql
SELECT COUNT(*) FROM mobility.silver.trips_clean;
SELECT reject_reason, COUNT(*) FROM mobility.silver.trips_quarantine GROUP BY reject_reason;

reject_reason,COUNT(*)
NEGATIVE_FARE,37444
DROPOFF_NOT_AFTER_PICKUP,870


In [0]:
bronze = spark.table("mobility.bronze.yellow_trips_raw").count()
clean = spark.table("mobility.silver.trips_clean").count()
bad = spark.table("mobility.silver.trips_quarantine").count()
src = spark.read.parquet("abfss://landing-files@proj2storage.dfs.core.windows.net/yellow_taxi/").count()

print("bronze:", bronze, " clean:", clean, " bad:", bad)
print("clean + bad == bronze:", clean + bad == bronze)
print("source files:", src, " bronze:", bronze, " same:", src == bronze)

bronze: 9554778  clean: 9415401  bad: 139377
clean + bad == bronze: True
source files: 9554778  bronze: 9554778  same: True
